# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
import os
import shutil
from datetime import datetime
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
print(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
print(f"✅ Torch CUDA available: {cuda_test}")
device_name_gpu = torch.cuda.get_device_name(0)
device_gpu = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device Name: {device_name_gpu} | Device reference: {device_gpu}")

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np
from PIL import Image
from pathlib import Path

# modelisation
from sklearn.metrics import accuracy_score
import torch.optim as optim
from torch.utils.tensorboard.writer import SummaryWriter
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms, models
from torchvision.models import MobileNet_V2_Weights
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from captum.attr import LayerGradCam, Occlusion

# Pour la visualisation des performances
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# 2. Loading and Data Enrichment

In [ ]:
def classer_images_par_age(
    repertoire_images,
    age_min=None,
    age_max=None
):
    """
    Organise les images dans des sous-dossiers en fonction de l'âge extrait du nom de fichier.
    - Peut filtrer une plage d'âges : age_min à age_max
    - Réindexe les âges sélectionnés en commençant à 0
    """

    ages_valides = set()

    # Étape 1 — Parcourir tous les fichiers et collecter les âges valides
    fichiers_eligibles = []
    for racine, _, fichiers in os.walk(repertoire_images):
        for fichier in fichiers:
            if fichier.lower().endswith(".jpg"):
                try:
                    age = int(fichier.split("_")[0])
                    if ((age_min is None or age >= age_min) and
                        (age_max is None or age <= age_max)):
                        fichiers_eligibles.append((fichier, age, racine))
                        ages_valides.add(age)
                except Exception as e:
                    print(f"⚠️ Ignoré : {fichier} (erreur : {e})")

    # Étape 2 — Réindexer les âges valides
    ages_valides = sorted(ages_valides)
    mapping_ages = {age: idx for idx, age in enumerate(ages_valides)}
    print(f"🎯 Mapping des âges : {mapping_ages}")

    # Étape 3 — Déplacer les fichiers vers les bons dossiers (réindexés)
    for fichier, age, racine in fichiers_eligibles:
        nouvelle_classe = str(mapping_ages[age])
        dest_dir = os.path.join(repertoire_images, nouvelle_classe)
        os.makedirs(dest_dir, exist_ok=True)

        chemin_source = os.path.join(racine, fichier)
        chemin_destination = os.path.join(dest_dir, fichier)

        try:
            shutil.move(chemin_source, chemin_destination)
        except Exception as e:
            print(f"❌ Erreur déplacement {fichier} : {e}")

    print("✅ Organisation terminée.")

In [ ]:
# Définir les transformations (optionnel, mais recommandé)
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Redimensionne les images
    transforms.ToTensor(),  # Convertit les images en tenseurs
])

In [ ]:
# dataset provenant de https://susanqq.github.io/UTKFace/ et décompressé localement avec application d'une reconstruction
# des sous-répertoires par classe avec la fonction utilitaire classer_images_par_age dans le but d'être compatible 
# avec image_dataset_from_directory
# Ajuster le nom du répertoire ou sont décompressés les images de visage et selectionner la plage pour les Ages en fonction
# des performance de votre machine (ajusté pour GPU RTX3080 ici)
data_dir = "C:\\Users\\remyc\\Downloads\\visages\\"  
min_age = 25
max_age = 45
classer_images_par_age(data_dir, min_age, max_age)

# Charger les données à partir du dossier
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

In [ ]:
# Cache solution 1 (Pré-chargement complet en mémoire d'un coup)
class ListDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

cached_train_ds = list(DataLoader(
    train_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True
))
cached_test_ds = list(DataLoader(
    test_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True
))
train_dl = DataLoader(
    ListDataset(cached_train_ds), batch_size=None, shuffle=True
)
test_dl = DataLoader(
    ListDataset(cached_test_ds), batch_size=None, shuffle=True
)

In [ ]:
# # Cache solution 2 (lazy loading progressif en mémoire)
# class CachedDataset(Dataset):
#     def __init__(self, base_dataset):
#         self.base = base_dataset
#         self.cache = {}

#     def __len__(self):
#         return len(self.base)

#     def __getitem__(self, idx):
#         if idx not in self.cache:
#             self.cache[idx] = self.base[idx]
#         return self.cache[idx]

# train_dl = DataLoader(
#     CachedDataset(train_ds), batch_size=32, shuffle=True,
#     num_workers=0, pin_memory=True
# )
# test_dl = DataLoader(
#     CachedDataset(test_ds), batch_size=32, shuffle=True,
#     num_workers=0, pin_memory=True
# )

In [ ]:
# Définitions des dimensions d'entrée et de sortie optimisées (nb : pour de la regression linéaire on n'utilisera pas le nombre de classes)
X_train_first, y_train_first = next(iter(train_dl))
shape_image = X_train_first.shape[1:]
print("Shape des images train :", shape_image)
num_classes = len(y_train_first)
print("Nombre de classes train:", num_classes)

X_test_first, y_test_first = next(iter(test_dl))
shape_image = X_test_first.shape[1:]
print("Shape des images test :", shape_image)  # type: ignore
num_classes = len(y_test_first)
print("Nombre de classes test :", num_classes)

# 3. Deep learning

## 3.1 Modèle basé sur Layer PyTorch (transfert learning)

#### Creation & Execution

In [ ]:
# récupération des poids ajusté d'un modèle MobileNet_V2
weights_mnetv2 = MobileNet_V2_Weights.DEFAULT
# récupération de la fonction de preprocessing du modèle pré entrainé
preprocess_mnetv2 = weights_mnetv2.transforms()
# initialisation de notre modèle basé sur ces poids
nn_pytorch_tl = models.mobilenet_v2(weights=weights_mnetv2)
# # gel des paramètres du modèle hérité
# for param in nn_pytorch_tl.parameters():
#     param.requires_grad = False
# Ajustement du modèle (dernière couche ici pour prendre en compte le nombre de classes)
nn_pytorch_tl.classifier = torch.nn.Sequential(
    torch.nn.Linear(1280, 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(256, num_classes)
)
# Chargement des poids du modèle si le fichier existe
MODEL_PATH = "nn_pytorch_tl.pth"
if os.path.exists(MODEL_PATH):
    print(f"🔁 Chargement des poids du modèle depuis {MODEL_PATH} pour continuer l'entrainement")
    nn_pytorch_tl.load_state_dict(torch.load(MODEL_PATH))
# Definition du device (GPU et CPU en fallback)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé:", device)
nn_pytorch_tl.to(device)
# Affichage
display(nn_pytorch_tl)

#### Entraînement et métriques

In [ ]:
# Initialisation du writer TensorBoard avec horodatage pour différencier les runs
log_dir = f"../../runs/nn_pytorch_tl_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)
example_batch = next(iter(train_dl))[0].to(device).float()
writer.add_graph(nn_pytorch_tl, example_batch)
# Nombre d'époques (itération de descente de gradient)
nb_epoch = 2
# Fonction de perte
loss_func = torch.nn.CrossEntropyLoss()
# Définition de l'optimizer
optimizer = optim.Adam(nn_pytorch_tl.parameters(), 1e-3)
# Définir le scheduler ReduceLROnPlateau
lf_plateau = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

for epoch in range(nb_epoch):
    # Bascule en mode entrainement
    nn_pytorch_tl.train()
    loss_total = 0
    # Barre de progression
    progress_bar = tqdm(
        train_dl, desc=f"Epoch {epoch+1:1d}", leave=True, disable=False
    )

    # Entrainer sur chaque batch
    for i, batch in enumerate(progress_bar):
        # Récupération des info du Batch de données
        X_batch, y_batch = batch
        # Affectation au Device
        X_batch = preprocess_mnetv2(X_batch.to(device))
        y_batch = y_batch.to(device)
        # Remise à zéro du Gradient
        nn_pytorch_tl.zero_grad()
        # Calcul de prédiction
        y_pred = nn_pytorch_tl(X_batch.to(torch.float32))
        # Calcul de la fonction de perte
        loss = loss_func(y_pred, y_batch) 
        # Calculer le gradient de la fonction de perte pour chaque couche
        loss.backward()
        # # Clipper le gradient entre 0 et 1 pour plus de stabilité
        # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        # Descente de gradient avec actualisation des paramètres
        optimizer.step()
        # Accumulation sur la perte totale
        loss_total += loss.item()
        train_loss_batch = loss_total/(i+1)
        # mise à jour de la progression
        progress_bar.set_postfix({
            "Batch train loss": f"{train_loss_batch:.3f}"
        })
    train_loss_epoch = loss_total/len(train_dl)

    # Évaluer sur l'ensemble de test
    nn_pytorch_tl.eval()
    test_loss_epoch = 0.0
    with torch.no_grad():
        for X_test, y_test in test_dl:
            X_test, y_test = preprocess_mnetv2(X_test.to(device)), y_test.to(device)
            y_pred = nn_pytorch_tl(X_test.to(torch.float32))
            loss = loss_func(y_pred, y_test)
            test_loss_epoch += loss.item()
    test_loss_epoch /= len(test_dl)

    # Mise à jour du scheduler
    lf_plateau.step(test_loss_epoch)

    # Affichage de la perte à la fin de l'époque
    print(f"Epoch {epoch+1}/{nb_epoch}, Train Loss: {train_loss_epoch:.3f}, Test Loss: {test_loss_epoch:.3f}")

    # Ajout de la loss à TensorBoard
    writer.add_scalar("Loss/train", train_loss_epoch, epoch + 1)
    writer.add_scalar("Loss/test", test_loss_epoch, epoch + 1)
    
# Libération du writer pour alimenter tensorboard
writer.close()

#### Prédiction et évaluation

In [ ]:
def evaluate(model, dataloader, criterion, device):
    # Passer en mode évaluation du modèle
    model.eval()
    # Initialiser la perte totale / predictions / valeurs réelles
    loss_val_total = 0
    pred_vals, true_vals = [], []
    # Parcourir tous les batches pour récupérer les prédictions/valeurs réelles
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        X_batch = preprocess_mnetv2(X_batch.to(device))
        y_batch = y_batch.to(device)
        # Pas de calcul de gradient nécessaire pendant la prédiction
        with torch.no_grad():
            y_pred = model(X_batch.to(torch.float32))
            # les prédictions sont une probabilité par classe on veut récupérer aussi les labels pour comparaison
            y_pred_label = torch.argmax(y_pred, dim=1)
        # Calcul de la perte pour ce batch
        loss = criterion(y_pred, y_batch)
        loss_val_total += loss.item()
        # Conversion en numpy et stockage pour le calcul des métriques (scikit ne tolère pas les tensors)
        y_pred_label_np = y_pred_label.detach().cpu().numpy()
        y_true_np = y_batch.cpu().numpy()
        pred_vals.append(y_pred_label_np)
        true_vals.append(y_true_np)
    # Concaténer tous les batches
    pred_vals = np.concatenate(pred_vals).ravel()
    true_vals = np.concatenate(true_vals).ravel()
    # Calcul de la moyenne de la loss
    loss_val_avg = loss_val_total / len(dataloader)
    # Calcul des métriques
    metrics = {
        "accuracy": accuracy_score(true_vals, pred_vals),
    }
    return loss_val_avg, metrics, pred_vals, true_vals

loss, metrics, y_test_pred_class, y_test_class = evaluate(
    model=nn_pytorch_tl,
    dataloader=test_dl,
    criterion=torch.nn.CrossEntropyLoss(),
    device=device,
)

print(f"Loss: {loss:.4f}")
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}")

In [ ]:
# Calculer la matrice de confusion
cnf_matrix = confusion_matrix(y_test_class, y_test_pred_class, normalize='true')
# Tracer la heatmap de la matrice de confusion
plt.figure(figsize=(10, 8))
plt.title("Matrice de confusion")
sns.heatmap(cnf_matrix, cmap='Blues', annot=True, cbar=False, fmt=".2f")
ages = [i + min_age for i in range(max_age - min_age + 1)]
plt.xticks(ticks=np.arange(len(ages)) + 0.5, labels=ages)
plt.yticks(ticks=np.arange(len(ages)) + 0.5, labels=ages)
plt.xlabel('Labels prédits')
plt.ylabel('Vrais labels')
plt.show()
print("Rapport de classification complet:\n", classification_report(y_test_class, y_test_pred_class, zero_division=0))

#### Sauvegarde du modèle

In [ ]:
torch.save(nn_pytorch_tl.state_dict(), MODEL_PATH)

# 4. Interprétabilité

#### Chargement d'une image unitaire et fonctions utilitaires

In [ ]:
def deprocess_tensor_for_display(tensor_img, mean, std):
    """Dénormalise un tenseur d'image [3, H, W] et le convertit pour affichage"""
    if tensor_img.ndim != 3 or tensor_img.shape[0] != 3:
        raise ValueError("L'image doit être un tensor [3, H, W]")

    # Convertir mean/std en tensors et broadcast
    mean = torch.tensor(mean).view(-1, 1, 1)
    std = torch.tensor(std).view(-1, 1, 1)

    # Dénormaliser
    img = tensor_img * std + mean
    img = img.clamp(0, 1)  # pour éviter les dépassements

    # Convertir en format [H, W, C] pour affichage
    img = img.permute(1, 2, 0).cpu().numpy()
    return img

In [ ]:
def show_importance(model, interpretor, input_tensor, target=0, device="cuda", **attribute_kwargs):
    input_tensor = input_tensor.to(device).unsqueeze(0)  # [1, C, H, W]
    model.to(device)
    model.eval()

    attributions = interpretor.attribute(inputs=input_tensor, target=target, **attribute_kwargs)
    heatmap = attributions.sum(dim=1).squeeze(0)  # [H, W]

    heatmap = heatmap.cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) != 0:
        heatmap /= np.max(heatmap)

    return heatmap  # [H, W] float in [0, 1]

In [ ]:
img_paths = [
    Path("C:/Users/remyc/Downloads/visage_perso/25_remy.jpg"),
    Path("C:/Users/remyc/Downloads/visage_perso/44_remy.jpg")
]
weights_mnetv2 = MobileNet_V2_Weights.DEFAULT
preprocess_mnetv2 = weights_mnetv2.transforms()
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Charger, prétraiter et empiler les images
img_tensors = []
for img_path in img_paths:
    img = Image.open(img_path).convert("RGB")  # convert() au cas où l’image est en niveaux de gris ou RGBA
    img_tensor = preprocess_mnetv2(img)  # shape : [3, 224, 224]
    img_tensors.append(img_tensor)

# Créer un batch tensoriel : [batch_size, 3, 224, 224]
batch_tensor = torch.stack(img_tensors)

# Affichage debug
print(f"Batch shape : {batch_tensor.shape}")  # ex: torch.Size([2, 3, 224, 224])

#### Grad-Cam

In [ ]:
# Sélectionner la couche cible dans MoblieNet_V2
target_layer = nn_pytorch_tl.features[18][0]  # Conv2d last layer
# Initialiser Grad-CAM
gradcam = LayerGradCam(forward_func=nn_pytorch_tl, layer=target_layer)

# 📊 Affichage côte à côte
fig, axes = plt.subplots(nrows=len(batch_tensor), ncols=2, figsize=(8, 4 * len(batch_tensor)))

if len(batch_tensor) == 1:
    axes = [axes]  # rendre 2D si un seul exemple

for i, ax_row in enumerate(axes):
    input_tensor = batch_tensor[i]
    heatmap = show_importance(nn_pytorch_tl, gradcam, 
                              input_tensor, target=0,  # 0 = classe 25 ans
                              device=str(device_gpu))

    # Image dénormalisée
    img_display = deprocess_tensor_for_display(input_tensor, IMAGENET_MEAN, IMAGENET_STD)

    # 📷 Image originale
    ax_row[0].imshow(img_display)
    ax_row[0].set_title(f"Image d'origine {i}")
    ax_row[0].axis("off")

    # 🔥 Heatmap seule (pas superposée ici)
    ax_row[1].imshow(heatmap, cmap="jet")
    ax_row[1].set_title(f"Grad-CAM (target=25 ans) {i}")
    ax_row[1].axis("off")

plt.tight_layout()
plt.show()

#### Occlusion

In [ ]:
# Initialiser l'occlusion
occlusion = Occlusion(forward_func=nn_pytorch_tl)
sliding_window_shapes=(3, 15, 15)
strides = (3, 8, 8)

# 📊 Affichage côte à côte
fig, axes = plt.subplots(nrows=len(batch_tensor), ncols=2, figsize=(8, 4 * len(batch_tensor)))

if len(batch_tensor) == 1:
    axes = [axes]  # rendre 2D si un seul exemple

for i, ax_row in enumerate(axes):
    input_tensor = batch_tensor[i]
    heatmap = show_importance(nn_pytorch_tl, occlusion, 
                              input_tensor, target=0,  # 0 = classe 25 ans 
                              device=str(device_gpu),
                              strides=strides, sliding_window_shapes=sliding_window_shapes)

    # Image dénormalisée
    img_display = deprocess_tensor_for_display(input_tensor, IMAGENET_MEAN, IMAGENET_STD)

    # 📷 Image originale
    ax_row[0].imshow(img_display)
    ax_row[0].set_title(f"Image d'origine {i}")
    ax_row[0].axis("off")

    # 🔥 Heatmap seule (pas superposée ici)
    ax_row[1].imshow(heatmap, cmap="jet")
    ax_row[1].set_title(f"Grad-CAM (target=25 ans) {i}")
    ax_row[1].axis("off")

plt.tight_layout()
plt.show()